# Amazon SageMaker Asynchronous Inference

In [25]:
import sagemaker
import boto3
from time import gmtime, strftime
from datetime import datetime
import time
import json
import os

boto_session = boto3.session.Session()
sm_session = sagemaker.session.Session()
sm_client = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")
region = boto_session.region_name
s3_client = boto3.client("s3", region)

today = datetime.now().strftime("%Y-%m-%d")

# Configuration
s3_bucket = 'textclassificationmldemo-model-archiving-us-east-1-2667'
bucket_prefix = 'models/model-a'
input_prefix = f'{bucket_prefix}/input/processed_json'
output_prefix = f'{bucket_prefix}/output/{today}'
print(input_prefix)

models/model-a/input/processed_json


## Create Model
Specifies the location of the pre-trained model stored in S3. 

In [7]:
from sagemaker import image_uris

model_s3_key = f"{s3_bucket}/{bucket_prefix}/model/model.tar.gz"
model_url = f"s3://{s3_bucket}/{model_s3_key}"
print(f"Uploading Model to {model_url}")

sm_role = sagemaker.get_execution_role()

# Specify an AWS container image and region as desired
container = image_uris.retrieve(region=region, framework="blazingtext", version="1")
print(container)

Uploading Model to s3://textclassificationmldemo-model-archiving-us-east-1-2667/textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/model/model.tar.gz
811284229777.dkr.ecr.us-east-1.amazonaws.com/blazingtext:1


#### Run this command if model is not available/created before

In [ ]:
# model_name = "TextClassification-ModelA"
# create_model_response = sm_client.create_model(
#     ModelName=model_name,
#     ExecutionRoleArn=sm_role,
#     PrimaryContainer={
#         "Image": container,
#         "ModelDataUrl": model_url,
#     },
# )

# print(f"Created Model: {create_model_response['ModelArn']}")

## Create EndpointConfig
Create the endpoint config if not created otherwise skip this cell. 

In [8]:
# Create EndpointConfig with async settings
endpoint_config_name = "TextClassificationMLDemo-TextClassification-Config"
model_name = "TextClassificationMLDemo-Model-A-20241218a-Model"

create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "ModelName": model_name,
            "InstanceType": "ml.m5.2xlarge",
            "InitialInstanceCount": 1,
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {
            "S3OutputPath": f"s3://{s3_bucket}/{bucket_prefix}/{output_prefix}",
        },
        "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 4},
    },
)
print(f"Created EndpointConfig: {create_endpoint_config_response['EndpointConfigArn']}")

Created EndpointConfig: arn:aws:sagemaker:us-east-1:266735847556:endpoint-config/TextClassificationMLDemo-TextClassification-Config


## Create Async Endpoint
Skip this if already endpoint created.

In [9]:
# Create Async Endpoint using the same endpoint config name
endpoint_name = "Text-Classification-Model-Archiving"

create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name, 
    EndpointConfigName=endpoint_config_name  # use the same name here
)

print(f"Created Endpoint: {create_endpoint_response['EndpointArn']}")

Created Endpoint: arn:aws:sagemaker:us-east-1:266735847556:endpoint/Text-Classification-Model-Archiving


------

## Invoke Endpoint
Invoke endpoint with multiple json files.

In [37]:
endpoint_name = 'TextClassificationMLDemo-TextClassification-Endpoint'
# endpoint_name = 'Text-Classification-Model-Archiving'
INPUT_S3_PREFIX = f's3://{s3_bucket}/{input_prefix}'
print(INPUT_S3_PREFIX)

def list_files_by_pattern(s3_prefix, max_files=1000):
    """
    Given an S3 prefix (e.g. "s3://bucket/key/"), iterate over indices
    to check for files named "input_0.json", "input_1.json", etc.
    Stops when a file is not found.
    Returns a list of full S3 URIs.
    """
    # Parse bucket and key prefix.
    parts = s3_prefix.replace("s3://", "").split("/", 1)
    bucket = parts[0]
    key_prefix = parts[1] if len(parts) > 1 else ""
    if not key_prefix.endswith("/"):
        key_prefix += "/"
    
    file_list = []
    for i in range(max_files):
        key = f"{key_prefix}input_{i}.json"
        try:
            # Use head_object to check if file exists.
            s3_client.head_object(Bucket=bucket, Key=key)
            file_uri = f"s3://{bucket}/{key}"
            file_list.append(file_uri)
            print(f"[{datetime.now()}] Found file: {file_uri}")
        except s3_client.exceptions.ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == '404':
                print(f"[{datetime.now()}] No file found for key: {key}. Ending search.")
                break
            else:
                print(f"[{datetime.now()}] Error checking key {key}: {e}")
                break
    return file_list


s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json


In [38]:

def invoke_async_for_file(endpoint_name, input_file):
    """
    Invokes the asynchronous endpoint for a single input file.
    """
    print(f"[{datetime.now()}] Invoking async endpoint for file: {input_file}")
    response = sm_runtime.invoke_endpoint_async(
        EndpointName=endpoint_name,
        InputLocation=input_file,
        ContentType="application/jsonlines",  # each file contains a single JSON object
        Accept="application/jsonlines"
    )
    inference_id = response.get("InferenceId")
    print(f"[{datetime.now()}] InferenceId for {input_file}: {inference_id}")
    return inference_id


In [39]:
def main():
    # Discover files by pattern.
    input_files = list_files_by_pattern(INPUT_S3_PREFIX)
    if not input_files:
        print(f"No input files found in {INPUT_S3_PREFIX}")
        return

    invocation_ids = {}
    start_time = time.time()
    
    # Loop through each discovered file and invoke the endpoint.
    for file in input_files:
        inference_id = invoke_async_for_file(endpoint_name, file)
        invocation_ids[file] = inference_id
        time.sleep(1)  # optional delay between invocations

    elapsed = time.time() - start_time
    print(f"\nSubmitted {len(invocation_ids)} async requests in {elapsed:.2f} seconds.")
    print("The async endpoint will write outputs to the configured S3 output location.")
    

main()

[2025-03-03 14:54:43.370045] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_0.json
[2025-03-03 14:54:43.383862] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_1.json
[2025-03-03 14:54:43.395874] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_2.json
[2025-03-03 14:54:43.412014] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_3.json
[2025-03-03 14:54:43.432582] No file found for key: models/model-a/input/processed_json/input_4.json. Ending search.
[2025-03-03 14:54:43.432642] Invoking async endpoint for file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_0.json
[2025-03-03 14:54:43.537691] InferenceId for s3://textclassificationmldemo-model-archiving-us-east-1-2667/